## How to build an interactive chloropleth map of France using plotly. 

### Step 1: Run all imports

In [1]:
import plotly as plt
import plotly.express as px
import json
from urllib.request import urlopen 
import pandas as pd

### Step 2: Load in and clean your dataset of choice for use. This tutorial uses data about reported Coronavirus cases in France in early 2020.

In [2]:
data = pd.read_csv('patient.csv')
data.head()

,id,sex,birth_year,country,region,departement,city,group,infection_reason,infection_order,infected_by,contact_number,confirmed_date,released_date,deceased_date,status,health,source,comments
0,NaN,male,NaN,France,Nouvelle-Aquitaine,Charente-Maritime,NaN,NaN,contact with patient,NaN,Creil patient,NaN,2020-02-28,2020-02-03,NaN,released,NaN,CP ARS Nouvelle-Aquitaine,NaN
1,NaN,male,NaN,France,Nouvelle-Aquitaine,Gironde,NaN,NaN,visit to Italy,NaN,NaN,NaN,2020-02-28,NaN,NaN,hospital,good,CP ARS Nouvelle-Aquitaine,NaN
2,NaN,female,NaN,France,Nouvelle-Aquitaine,Landes,NaN,NaN,contact with patient,NaN,Creil patient,NaN,2020-02-28,NaN,NaN,hospital,good,CP ARS Nouvelle-Aquitaine,NaN
3,NaN,male,NaN,France,Nouvelle-Aquitaine,Pyrénées-Atlantiques,NaN,NaN,visit to Italy,NaN,NaN,NaN,2020-03-03,2020-03-06,NaN,released,good,CP ARS Nouvelle-Aquitaine,NaN
4,NaN,male,NaN,France,Nouvelle-Aquitaine,Lot-et-Garonne,NaN,Mulhouse religious gathering,visit to Mulhouse religious gathering,NaN,NaN,NaN,2020-03-05,NaN,NaN,home isolation,good,CP ARS Nouvelle-Aquitaine,NaN


#### Step 2.a: If necessary, group the regions together to obtain one value per region.

In [3]:
regions = data.groupby('region', as_index=False).size() # counts the amount of cases per region and creates a column
regions

,region,size
0,Auvergne-Rhône-Alpes,228
1,Bourgogne-Franche-Comté,165
2,Bretagne,157
3,Centre-Val de Loire,20
4,Corse,63
5,Grand-Est,461
6,Guadeloupe,1
7,Guyane,6
8,Hauts-de-France,49
9,Ile-de-France,440


### Step 3: Plotly does not have predetermined geometries for France or any other country outside the US. A geojson file containing the geometries of French regions has to be loaded instead. 

In [10]:
with urlopen('https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/france-regions.geojson') as response:
 France = json.load(response) 
# loads geojson data (latitudes and longitudes of regions)
# can be switched to match country of choice by changing what comes immediately before the .geojson portion of the url

#### Step 3.a: ID each region based on their regional name. Check to see what regions the geojson file contains for France.

In [5]:
for feature in France['features']:
    feature['id'] = feature['properties']['name']

In [6]:
for feature in France['features']:
    print(feature['properties'].get('name'))

Alsace
Lorraine
Corsica
Brittany
Lower Normandy
Upper Normandy
Champagne-Ardenne
Burgundy
Pays-de-la-Loire
Limousin
Midi-Pyrenées
Franche-Comté
Provence-Alpes-Cote d'Azur
Rhone-Alpes
Picardy
Aquitaine
Centre
Ile-de-France
Nord-Pas de Calais
Poitou-Charentes
Auvergne
Languedoc-Roussilion


### Step 4: If necessary, change region naming in dataset to be consistent with geojson mappings.
#### France combined regions together in 2016, but the available geojson file contains the uncombined region name. The dataset used contains the combined region names. 

In [7]:
mapping = {"Auvergne-Rhône-Alpes": ["Auvergne", "Rhone-Alpes"],
    "Bourgogne-Franche-Comté": ["Burgundy", "Franche-Comté"],
    "Bretagne": ["Brittany"],
    "Centre-Val de Loire": ["Centre"],
    "Corse": ["Corsica"],
    "Grand-Est": ["Alsace", "Lorraine", "Champagne-Ardenne"],
    "Hauts-de-France": ["Nord-Pas de Calais", "Picardy"],
    "Ile-de-France": ["Ile-de-France"],
    "Normandie": ["Lower Normandy", "Upper Normandy"],
    "Nouvelle-Aquitaine": ["Aquitaine", "Limousin", "Poitou-Charentes"],
    "Occitanie": ["Languedoc-Roussillon", "Midi-Pyrenees"],
    "Pays-de-la-Loire": ["Pays-de-la-Loire"],
    "PACA": ["Provence-Alpes-Cote d'Azur"]} # breaks down each region in dataset to corresponding geojson file region

regions['old_regions'] = regions['region'].map(mapping) # applies mapping
regions['old_regions'] = regions.apply(
    lambda row: row['old_regions'] if isinstance(row['old_regions'], list) else [row['region']],
    axis=1) # makes every row into a list so further cleaning can be done
regions['old_regions_count'] = regions['old_regions'].apply(len) 
regions_expanded = regions.explode('old_regions')
regions_expanded['size'] = regions_expanded['size'] / regions_expanded['old_regions_count'] 
# breaks down covid cases across previously combined regions


### Step 5: Change column headers to the labels desired for visualization.   

In [8]:
regions_expanded = regions_expanded.rename(columns={'old_regions': 'Region', 'size':'Reported Cases'}) 

### Step 6: Create choropleth using plotly! 

In [9]:
fig = px.choropleth(regions_expanded, # add dataset used
                    geojson=France, # adding geojson mapping 
                    locations='Region', # location feature of dataset to be used
                    featureidkey='properties.name', # id of geojson to be used
                    color='Reported Cases', # what to base shading on
                    hover_name='Region', # interactivity element
                    color_continuous_scale='pinkyl', # color map for visual
                    title="Early Pandemic Coronavirus Cases in France")

fig.update_geos(fitbounds="locations", # sets zoom in of chloropleth so only desired locations are in view
                visible=False)# removes all other outlines
fig.write_html('France_Choropleth.html') # creates html file